In [ ]:
import os
import json
import cv2

# =========================================================
# 원본 데이터셋 경로
# =========================================================
DATASET_DIR = r"C:\Users\KonYang\Desktop\dataset"

# =========================================================
# crop 저장 경로
# =========================================================
SAVE_DIR = r"C:\Users\KonYang\Desktop\cropped_dataset_more"

os.makedirs(SAVE_DIR, exist_ok=True)

# =========================================================
# train / val / test 반복
# =========================================================
for split in ["train", "val", "test"]:

    split_path = os.path.join(DATASET_DIR, split)

    # train / val / test 폴더 없으면 skip
    if not os.path.exists(split_path):
        print(f"{split} 폴더 없음")
        continue

    print(f"\n=========================")
    print(f"{split} 처리 시작")
    print(f"=========================")

    # =========================================================
    # 클래스 반복
    # =========================================================
    for cls_name in os.listdir(split_path):

        class_path = os.path.join(split_path, cls_name)

        if not os.path.isdir(class_path):
            continue

        print(f"\n[{split}] 클래스: {cls_name}")

        # =========================================================
        # 저장 폴더 생성
        # =========================================================
        save_img_dir = os.path.join(
            SAVE_DIR,
            split,
            "crop_img",
            cls_name
        )

        save_json_dir = os.path.join(
            SAVE_DIR,
            split,
            "crop_json",
            cls_name
        )

        os.makedirs(save_img_dir, exist_ok=True)
        os.makedirs(save_json_dir, exist_ok=True)

        # =========================================================
        # 파일 반복
        # =========================================================
        file_list = os.listdir(class_path)

        for idx, file_name in enumerate(file_list):

            # jpg 파일만 처리
            if not file_name.lower().endswith(".jpg"):
                continue

            print(f"[{idx}] 처리중: {file_name}")

            # =========================================================
            # 이미지 경로
            # =========================================================
            img_path = os.path.join(class_path, file_name)

            # =========================================================
            # json 경로
            # =========================================================
            base_name = os.path.splitext(file_name)[0]

            json_name = base_name + ".json"

            json_path = os.path.join(class_path, json_name)

            # json 없으면 skip
            if not os.path.exists(json_path):
                print(f"JSON 없음: {json_path}")
                continue

            # =========================================================
            # 이미지 읽기
            # =========================================================
            image = cv2.imread(img_path)

            if image is None:
                print(f"이미지 읽기 실패: {img_path}")
                continue

            # =========================================================
            # JSON 읽기
            # =========================================================
            try:
                with open(json_path, "r", encoding="utf-8") as f:
                    data = json.load(f)

            except Exception as e:
                print(f"JSON 읽기 실패: {json_name}")
                print(e)
                continue

            try:
                # =========================================================
                # box 찾기
                # =========================================================
                box_info = None

                for item in data["labelingInfo"]:

                    if "box" in item:

                        box_info = item["box"]["location"][0]
                        break

                # box 없으면 skip
                if box_info is None:
                    print(f"box 없음: {file_name}")
                    continue

                # =========================================================
                # 좌표 추출
                # =========================================================
                x = int(box_info["x"])
                y = int(box_info["y"])
                w = int(box_info["width"])
                h = int(box_info["height"])

                # =========================================================
                # padding 설정
                # =========================================================
                padding = 80

                img_h, img_w = image.shape[:2]

                x1 = max(0, x - padding)
                y1 = max(0, y - padding)

                x2 = min(img_w, x + w + padding)
                y2 = min(img_h, y + h + padding)

                # =========================================================
                # crop
                # =========================================================
                crop_img = image[y1:y2, x1:x2]

                # 빈 이미지 체크
                if crop_img.size == 0:
                    print(f"Crop 실패: {file_name}")
                    continue

                # =========================================================
                # crop 이미지 저장
                # =========================================================
                save_img_path = os.path.join(
                    save_img_dir,
                    file_name
                )

                success = cv2.imwrite(save_img_path, crop_img)

                if not success:
                    print(f"이미지 저장 실패: {save_img_path}")
                    continue

                # =========================================================
                # crop json 생성
                # =========================================================
                crop_json = {
                    "original_image": file_name,
                    "class": cls_name,
                    "crop_box": {
                        "x1": x1,
                        "y1": y1,
                        "x2": x2,
                        "y2": y2
                    }
                }

                # =========================================================
                # json 저장
                # =========================================================
                save_json_path = os.path.join(
                    save_json_dir,
                    json_name
                )

                with open(save_json_path, "w", encoding="utf-8") as f:
                    json.dump(
                        crop_json,
                        f,
                        indent=4,
                        ensure_ascii=False
                    )

                print(f"저장 완료: {save_img_path}")

            except Exception as e:
                print(f"오류 발생: {file_name}")
                print(e)

print("\n=========================")
print("전체 crop 완료!")
print("=========================")